# Section 1 — Motivation

Kalman filters, extended Kalman filters, unscented Kalman filters, simulations, and later 6DOF integrations all depend on a consistent state definition. A state vector is not just an array of numbers. The order of the states, their names, their units, and their grouped physical meaning affect every downstream matrix and model.

If this contract is ambiguous, later equations can run numerically while still being physically wrong. A matrix may have the right shape, a vector may have the right length, and a filter may produce numbers, while the implementation is quietly mixing position, velocity, attitude, or bias terms in the wrong places.

This notebook does not implement a filter. It builds the state-vector contract.

The state vector should store scalar components in exactly the order declared by the user. Semantic groups should sit on top of that scalar storage. Non-Euclidean quantities such as quaternions, MRPs, and Euler angles should be represented through group metadata, not through individual scalar components.

```text
Scalar components define storage.
Groups define meaning.
Representations define geometry.
```

This design is intended to feel ergonomic in Python while still being deterministic enough to map later to fixed-index C/C++ implementations. Python users can write expressive state declarations, while the architecture keeps the underlying numerical layout explicit and inspectable.


# Section 2 — Imports and Validation Style

This notebook keeps validation close to the class that owns the rule. Earlier drafts used several standalone helper functions such as `validate_non_empty_string`, `is_numeric_scalar`, `validate_state_name`, `validate_unique_names`, and `parse_state_field`.

Those helpers were correct, but they made this small architecture feel more abstract than it needed to be. In this version:

- `StateField` validates scalar component metadata directly.
- `StateGroup` validates group metadata directly.
- `StateVector` parses user-facing keyword inputs directly.
- `StateVector` composes `StateField` and `StateGroup`; it does not need a separate `parse_state_field` function.

This does duplicate a few lines of validation, but it keeps the notebook easier to read and makes each class responsible for its own boundary.


In [1]:
from dataclasses import dataclass
import keyword

import numpy as np


# Section 3 — `StateField`

`StateField` is the canonical internal representation of one scalar state component. It stores the scalar component name, numerical value, and unit.

It does not store index, group information, covariance, dynamics, or representation. Representation does not belong to `StateField`, because a scalar such as `q1` is not a quaternion by itself. A quaternion only becomes meaningful when several scalar components are interpreted together.


In [2]:
@dataclass(frozen=True)
class StateField:
    '''Canonical scalar state component with a name, float value, and unit.'''

    name: str
    value: float
    unit: str

    def __post_init__(self) -> None:
        '''Validate and normalize one scalar state component.'''
        # Names are public API, so they must be readable Python identifiers.
        if not isinstance(self.name, str):
            raise TypeError(
                f"StateField name must be a string; received {type(self.name).__name__}."
            )
        name = self.name.strip()
        if not name:
            raise ValueError("StateField name must be a non-empty string.")
        if not name.isidentifier():
            raise ValueError(
                f"StateField name must be a valid Python identifier; received {name!r}."
            )
        if keyword.iskeyword(name):
            raise ValueError(
                f"StateField name must not be a Python keyword; received {name!r}."
            )

        # State storage is real scalar storage, not containers or booleans.
        if isinstance(self.value, (bool, np.bool_)):
            raise TypeError(f"StateField value for {name!r} must not be bool.")
        if not isinstance(self.value, (int, float, np.integer, np.floating)):
            raise TypeError(
                f"StateField value for {name!r} must be a real numeric scalar; "
                f"received {type(self.value).__name__}."
            )
        if not np.isfinite(self.value):
            raise ValueError(f"StateField value for {name!r} must be finite.")

        if not isinstance(self.unit, str):
            raise TypeError(
                f"Unit for StateField {name!r} must be a string; "
                f"received {type(self.unit).__name__}."
            )
        unit = self.unit.strip()
        if not unit:
            raise ValueError(f"Unit for StateField {name!r} must be a non-empty string.")

        object.__setattr__(self, "name", name)
        object.__setattr__(self, "value", float(self.value))
        object.__setattr__(self, "unit", unit)


In [3]:
field_x = StateField("x", 1.0, "m")
field_vx = StateField("vx", 0.0, "m/s")

assert field_x.name == "x"
assert field_x.value == 1.0
assert field_x.unit == "m"
assert field_vx.unit == "m/s"

print("Valid StateField cases work.")


Valid StateField cases work.


In [4]:
try:
    StateField("", 1.0, "m")
except Exception as exc:
    print(f"Empty StateField name raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Empty StateField name should fail.")


Empty StateField name raises ValueError: StateField name must be a non-empty string.


In [5]:
try:
    StateField("x-position", 1.0, "m")
except Exception as exc:
    print(f"Invalid StateField name raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Invalid StateField name should fail.")


Invalid StateField name raises ValueError: StateField name must be a valid Python identifier; received 'x-position'.


In [6]:
try:
    StateField("class", 1.0, "m")
except Exception as exc:
    print(f"Python keyword StateField name raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Python keyword StateField name should fail.")


Python keyword StateField name raises ValueError: StateField name must not be a Python keyword; received 'class'.


In [7]:
try:
    StateField("x", "1.0", "m")
except Exception as exc:
    print(f"Non-numeric StateField value raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Non-numeric StateField value should fail.")


Non-numeric StateField value raises TypeError: StateField value for 'x' must be a real numeric scalar; received str.


In [8]:
try:
    StateField("x", True, "m")
except Exception as exc:
    print(f"Bool StateField value raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Bool StateField value should fail.")


Bool StateField value raises TypeError: StateField value for 'x' must not be bool.


In [9]:
try:
    StateField("x", 1.0, "")
except Exception as exc:
    print(f"Empty StateField unit raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Empty StateField unit should fail.")


Empty StateField unit raises ValueError: Unit for StateField 'x' must be a non-empty string.


# Section 4 — `StateGroup`

`StateGroup` is a semantic grouping of already-existing scalar components. Groups allow non-contiguous components to be interpreted together. Examples include `position`, `velocity`, `attitude`, `body_rate`, and `gyro_bias`.

A group stores component names, resolved integer indices, and representation metadata. For now, `representation` is only a string. In the future, `representation` may become a class that defines the physics and geometry of the representation.

This will matter for non-Euclidean quantities such as quaternions, MRPs, and Euler angles, where operations such as residuals, correction, normalization, and averaging are not ordinary Euclidean vector operations.

This is a composed architecture: scalar storage remains simple, while representation-specific behavior can be attached later without bloating `StateVector`.


In [10]:
@dataclass(frozen=True)
class StateGroup:
    '''Semantic view over existing scalar components and their indices.'''

    name: str
    components: tuple[str, ...]
    indices: tuple[int, ...]
    representation: str = "euclidean"

    def __post_init__(self) -> None:
        '''Validate group metadata and known representation dimensions.'''
        if not isinstance(self.name, str):
            raise TypeError(
                f"StateGroup name must be a string; received {type(self.name).__name__}."
            )
        name = self.name.strip()
        if not name:
            raise ValueError("StateGroup name must be a non-empty string.")
        if not name.isidentifier():
            raise ValueError(
                f"StateGroup name must be a valid Python identifier; received {name!r}."
            )
        if keyword.iskeyword(name):
            raise ValueError(
                f"StateGroup name must not be a Python keyword; received {name!r}."
            )

        if not isinstance(self.components, tuple):
            raise TypeError(f"Components for StateGroup {name!r} must be a tuple.")
        if not self.components:
            raise ValueError(f"StateGroup {name!r} must contain at least one component.")

        components: list[str] = []
        seen_components: set[str] = set()
        for position, component in enumerate(self.components):
            if not isinstance(component, str):
                raise TypeError(
                    f"Component {position} for StateGroup {name!r} must be a string; "
                    f"received {type(component).__name__}."
                )
            component_name = component.strip()
            if not component_name:
                raise ValueError(
                    f"Component {position} for StateGroup {name!r} must be non-empty."
                )
            if not component_name.isidentifier():
                raise ValueError(
                    f"Component {position} for StateGroup {name!r} must be a "
                    f"valid Python identifier; received {component_name!r}."
                )
            if keyword.iskeyword(component_name):
                raise ValueError(
                    f"Component {position} for StateGroup {name!r} must not be "
                    f"a Python keyword; received {component_name!r}."
                )
            if component_name in seen_components:
                raise ValueError(
                    f"StateGroup {name!r} has duplicate component {component_name!r}."
                )
            components.append(component_name)
            seen_components.add(component_name)

        if not isinstance(self.indices, tuple):
            raise TypeError(f"Indices for StateGroup {name!r} must be a tuple.")
        if not self.indices:
            raise ValueError(f"StateGroup {name!r} must contain at least one index.")

        indices: list[int] = []
        for position, index in enumerate(self.indices):
            if isinstance(index, (bool, np.bool_)) or not isinstance(index, (int, np.integer)):
                raise TypeError(
                    f"Index {position} for StateGroup {name!r} must be a "
                    f"non-negative integer; received {type(index).__name__}."
                )
            integer_index = int(index)
            if integer_index < 0:
                raise ValueError(
                    f"Index {position} for StateGroup {name!r} must be "
                    f"non-negative; received {integer_index}."
                )
            indices.append(integer_index)

        if len(components) != len(indices):
            raise ValueError(
                f"StateGroup {name!r} must have the same number of components "
                f"and indices; received {len(components)} components and "
                f"{len(indices)} indices."
            )

        if not isinstance(self.representation, str):
            raise TypeError(
                f"Representation for StateGroup {name!r} must be a string; "
                f"received {type(self.representation).__name__}."
            )
        representation = self.representation.strip()
        if not representation:
            raise ValueError(
                f"Representation for StateGroup {name!r} must be a non-empty string."
            )

        representation_key = representation.lower()
        dimension = len(components)
        if representation_key.startswith("quaternion") and dimension != 4:
            raise ValueError(
                f"Quaternion StateGroup {name!r} must have dimension 4; "
                f"received dimension {dimension}."
            )
        if representation_key == "mrp" and dimension != 3:
            raise ValueError(
                f"MRP StateGroup {name!r} must have dimension 3; "
                f"received dimension {dimension}."
            )
        if representation_key.startswith("euler") and dimension != 3:
            raise ValueError(
                f"Euler StateGroup {name!r} must have dimension 3; "
                f"received dimension {dimension}."
            )

        object.__setattr__(self, "name", name)
        object.__setattr__(self, "components", tuple(components))
        object.__setattr__(self, "indices", tuple(indices))
        object.__setattr__(self, "representation", representation)


In [11]:
position_group = StateGroup("position", ("x", "y"), (0, 2))
attitude_group = StateGroup(
    "attitude",
    ("q1", "q2", "q3", "q4"),
    (0, 1, 2, 3),
    "quaternion_jpl_scalar_last",
)

assert position_group.indices == (0, 2)
assert attitude_group.representation == "quaternion_jpl_scalar_last"

print("Valid StateGroup cases work.")


Valid StateGroup cases work.


In [12]:
try:
    StateGroup("", ("x",), (0,))
except Exception as exc:
    print(f"Empty StateGroup name raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Empty StateGroup name should fail.")


Empty StateGroup name raises ValueError: StateGroup name must be a non-empty string.


In [13]:
try:
    StateGroup("position", (), (0,))
except Exception as exc:
    print(f"Empty StateGroup components raise {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Empty StateGroup components should fail.")


Empty StateGroup components raise ValueError: StateGroup 'position' must contain at least one component.


In [14]:
try:
    StateGroup("position", ("x", "x"), (0, 1))
except Exception as exc:
    print(f"Duplicate StateGroup components raise {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Duplicate StateGroup components should fail.")


Duplicate StateGroup components raise ValueError: StateGroup 'position' has duplicate component 'x'.


In [15]:
try:
    StateGroup("position", ("x", "y"), (0,))
except Exception as exc:
    print(f"Component/index length mismatch raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Component/index length mismatch should fail.")


Component/index length mismatch raises ValueError: StateGroup 'position' must have the same number of components and indices; received 2 components and 1 indices.


In [16]:
try:
    StateGroup("attitude", ("q1", "q2", "q3"), (0, 1, 2), "quaternion_jpl_scalar_last")
except Exception as exc:
    print(f"Invalid quaternion group raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Invalid quaternion group should fail.")


Invalid quaternion group raises ValueError: Quaternion StateGroup 'attitude' must have dimension 4; received dimension 3.


In [46]:
try:
    StateGroup("attitude", ("s1", "s2", "s3", "s4"), (0, 1, 2, 3), "mrp")
except Exception as exc:
    print(f"Invalid MRP group raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Invalid MRP group should fail.")


Invalid MRP group raises ValueError: MRP StateGroup 'attitude' must have dimension 3; received dimension 4.


# Section 5 — `StateVector`

`StateVector` is the orchestrator. It is the main public class users interact with.

Users declare scalar components through keyword arguments. The order of keyword arguments defines the numerical state order. Each keyword input is internally normalized into a `StateField`.

`StateVector` owns the ordered fields, NumPy value array, name-to-index map, unit lookup, and groups. Groups are added later through `add_group`.

`StateGroup` is used inside `StateVector`: `add_group` is the registration command that resolves component names to indices and stores a `StateGroup`; `group` is the query method that returns the stored `StateGroup` metadata. These two methods are intentionally different: one mutates the state-vector definition, and the other reads from it.

Component names and group names must not conflict, because `x["name"]` should never be ambiguous. `StateVector` should automate bookkeeping and validation, but not infer physics.

This notebook does not expose public matrix validators. Matrix-shape contracts belong with later process and measurement model work.


In [18]:
class StateVector:
    '''Ordered scalar state container with units, groups, and copy-safe values.'''

    def __init__(self, **components: object) -> None:
        '''Create a state vector from keyword scalar components in declaration order.'''
        if not components:
            raise ValueError("StateVector requires at least one state component.")

        fields: list[StateField] = []
        seen_names: set[str] = set()
        for name, raw_value in components.items():
            # User input is intentionally simple: x=1.0 or x=(1.0, "m").
            if isinstance(raw_value, tuple):
                if len(raw_value) != 2:
                    raise ValueError(
                        f"State component {name!r} must be a numeric scalar or "
                        "a (value, unit) tuple."
                    )
                value, unit = raw_value
            else:
                value = raw_value
                unit = "unspecified"

            field = StateField(name=name, value=value, unit=unit)
            if field.name in seen_names:
                raise ValueError(
                    f"StateVector component names must be unique; "
                    f"duplicate name {field.name!r} found."
                )
            fields.append(field)
            seen_names.add(field.name)

        self._fields = tuple(fields)
        self._values = np.array([field.value for field in fields], dtype=float)
        self._name_to_index = {field.name: index for index, field in enumerate(fields)}
        self._units = {field.name: field.unit for field in fields}
        self._groups: dict[str, StateGroup] = {}

    @property
    def dimension(self) -> int:
        '''Return the number of scalar components.'''
        return len(self._fields)

    @property
    def names(self) -> tuple[str, ...]:
        '''Return scalar component names in numerical storage order.'''
        return tuple(field.name for field in self._fields)

    @property
    def units(self) -> tuple[str, ...]:
        '''Return scalar component units in numerical storage order.'''
        return tuple(field.unit for field in self._fields)

    @property
    def values(self) -> np.ndarray:
        '''Return a copy of the numerical state values.'''
        return self._values.copy()

    @property
    def groups(self) -> dict[str, StateGroup]:
        '''Return a shallow copy of registered groups by name.'''
        return dict(self._groups)

    def index(self, name: str) -> int:
        '''Return the integer index for one scalar component name.'''
        if not isinstance(name, str):
            raise TypeError(
                f"StateVector component name must be a string; received {type(name).__name__}."
            )
        lookup_name = name.strip()

        if lookup_name in self._name_to_index:
            return self._name_to_index[lookup_name]

        if lookup_name in self._groups:
            raise KeyError(
                f"{lookup_name!r} is a group, not a scalar component. "
                "Use indices(...) for group access."
            )

        raise KeyError(self._unknown_name_message(lookup_name))

    def indices(self, name: str) -> tuple[int, ...]:
        '''Return scalar or group indices as a tuple.'''
        if not isinstance(name, str):
            raise TypeError(
                f"StateVector name must be a string; received {type(name).__name__}."
            )
        lookup_name = name.strip()

        if lookup_name in self._name_to_index:
            return (self._name_to_index[lookup_name],)

        if lookup_name in self._groups:
            return self._groups[lookup_name].indices

        raise KeyError(self._unknown_name_message(lookup_name))

    def unit(self, name: str) -> str:
        '''Return the unit string for one scalar component.'''
        if not isinstance(name, str):
            raise TypeError(
                f"StateVector component name must be a string; received {type(name).__name__}."
            )
        lookup_name = name.strip()

        if lookup_name in self._units:
            return self._units[lookup_name]

        if lookup_name in self._groups:
            raise KeyError(
                f"Units are stored per scalar component; {lookup_name!r} is a group."
            )

        raise KeyError(self._unknown_name_message(lookup_name))

    def add_group(
        self,
        name: str,
        components: tuple[str, ...],
        representation: str = "euclidean",
    ) -> None:
        '''Register a semantic group over existing scalar components.'''
        candidate = StateGroup(
            name=name,
            components=components,
            indices=tuple(0 for _ in components) if isinstance(components, tuple) else (),
            representation=representation,
        )

        if candidate.name in self._name_to_index:
            raise ValueError(
                f"Group name {candidate.name!r} conflicts with an existing scalar component."
            )

        if candidate.name in self._groups:
            raise ValueError(f"Group name {candidate.name!r} already exists.")

        missing = tuple(
            component
            for component in candidate.components
            if component not in self._name_to_index
        )
        if missing:
            raise KeyError(
                f"Cannot create group {candidate.name!r}; unknown component(s) {missing}. "
                f"Available components: {self.names}."
            )

        group_indices = tuple(self._name_to_index[component] for component in candidate.components)
        self._groups[candidate.name] = StateGroup(
            name=candidate.name,
            components=candidate.components,
            indices=group_indices,
            representation=candidate.representation,
        )

    def group(self, name: str) -> StateGroup:
        '''Return the registered `StateGroup` with the requested name.'''
        if not isinstance(name, str):
            raise TypeError(
                f"StateVector group name must be a string; received {type(name).__name__}."
            )
        group_name = name.strip()

        if group_name in self._groups:
            return self._groups[group_name]

        if group_name in self._name_to_index:
            raise KeyError(f"{group_name!r} is a scalar component, not a group.")

        raise KeyError(self._unknown_name_message(group_name))

    def with_values(self, values: object) -> "StateVector":
        '''Return a new StateVector with the same metadata and new values.'''
        new_values = self._coerce_state_values(values, "values")
        replacement_components = {
            field.name: (new_values[index], field.unit)
            for index, field in enumerate(self._fields)
        }

        updated = StateVector(**replacement_components)
        for group in self._groups.values():
            updated.add_group(
                group.name,
                group.components,
                representation=group.representation,
            )

        return updated

    def summary(self) -> None:
        '''Print a compact human-readable summary of components and groups.'''
        print(f"StateVector dimension: {self.dimension}")
        print("Components:")
        for index, field in enumerate(self._fields):
            print(
                f"  [{index}] {field.name}: value={self._values[index]:g}, "
                f"unit={field.unit}"
            )

        if not self._groups:
            print("Groups: none")
            return

        print("Groups:")
        for group in self._groups.values():
            print(
                f"  {group.name}: components={group.components}, "
                f"indices={group.indices}, representation={group.representation}"
            )

    def __getitem__(self, key: object) -> float | np.ndarray:
        '''Read scalar values by name/index or grouped values by group name.'''
        if isinstance(key, str):
            if key in self._name_to_index:
                return float(self._values[self._name_to_index[key]])

            if key in self._groups:
                return self._values[list(self._groups[key].indices)].copy()

            raise KeyError(self._unknown_name_message(key))

        if isinstance(key, (bool, np.bool_)):
            raise TypeError("Boolean indexing is not supported by StateVector.")

        if isinstance(key, (int, np.integer)):
            integer_index = int(key)
            if integer_index < 0 or integer_index >= self.dimension:
                raise IndexError(
                    f"StateVector index {integer_index} is out of range for "
                    f"dimension {self.dimension}."
                )
            return float(self._values[integer_index])

        if isinstance(key, slice):
            return self._values[key].copy()

        raise TypeError(
            "StateVector indices must be component/group names, integers, or slices; "
            f"received {type(key).__name__}."
        )

    def _coerce_state_values(self, values: object, label: str) -> np.ndarray:
        '''Return a numeric array matching this vector's scalar shape.'''
        try:
            array = np.asarray(values)
        except ValueError as exc:
            raise TypeError(f"{label} must be convertible to a one-dimensional array.") from exc

        if array.shape != (self.dimension,):
            raise ValueError(
                f"{label} must have shape ({self.dimension},); received shape {array.shape}."
            )

        flat_values = []
        for position, value in enumerate(array.flat):
            if isinstance(value, (bool, np.bool_)):
                raise TypeError(f"{label} entry {position} must not be bool.")
            if not isinstance(value, (int, float, np.integer, np.floating)):
                raise TypeError(
                    f"{label} entry {position} must be a real numeric scalar; "
                    f"received {type(value).__name__}."
                )
            if not np.isfinite(value):
                raise ValueError(f"{label} entry {position} must be finite.")
            flat_values.append(float(value))

        return np.array(flat_values, dtype=float).reshape(array.shape)

    def _unknown_name_message(self, name: str) -> str:
        '''Build a clear error message for unknown component or group names.'''
        return (
            f"Unknown component or group {name!r}. "
            f"Available components: {self.names}. "
            f"Available groups: {tuple(self._groups)}."
        )


In [19]:
x = StateVector(
    x=(1.0, "m"),
    vx=(0.0, "m/s"),
    y=(2.0, "m"),
    vy=(0.0, "m/s"),
)

assert x.names == ("x", "vx", "y", "vy")
np.testing.assert_allclose(x.values, np.array([1.0, 0.0, 2.0, 0.0]))
assert x["vx"] == 0.0
assert x.unit("vx") == "m/s"

x.add_group("position", ("x", "y"))
assert isinstance(x.group("position"), StateGroup)
assert x.indices("position") == (0, 2)
np.testing.assert_allclose(x["position"], np.array([1.0, 2.0]))

try:
    x.add_group("x", ("x", "y"))
except Exception as exc:
    print(f"Group-name conflict raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Expected group-name conflict to fail.")

try:
    x["missing"]
except Exception as exc:
    print(f"Unknown name access raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Expected unknown name access to fail.")

x_updated = x.with_values(np.array([10.0, 1.0, 20.0, 2.0]))
assert x_updated.names == x.names
assert x_updated.units == x.units
assert x_updated.indices("position") == x.indices("position")
np.testing.assert_allclose(x_updated.values, np.array([10.0, 1.0, 20.0, 2.0]))
np.testing.assert_allclose(x.values, np.array([1.0, 0.0, 2.0, 0.0]))

print("StateVector behavior checks passed.")


Group-name conflict raises ValueError: Group name 'x' conflicts with an existing scalar component.
Unknown name access raises KeyError: "Unknown component or group 'missing'. Available components: ('x', 'vx', 'y', 'vy'). Available groups: ('position',)."
StateVector behavior checks passed.


# Section 6 — Design Checkpoint

This notebook has established:

- deterministic scalar ordering;
- internal normalized `StateField` objects;
- semantic grouping through `StateGroup`;
- representation metadata;
- non-contiguous group support;
- validation of names, units, scalar values, groups, and replacement values;
- an explicit state contract.

It has deliberately not implemented:

- filter prediction or update;
- covariance ownership;
- process models;
- measurement models;
- automatic `F`, `Q`, `H`, or `R` generation;
- public vector or matrix validators for model contracts;
- non-Euclidean operations;
- a simulation runner;
- plotting;
- C++ code generation.


# Section 7 — Examples

The examples below show that the architecture does not prescribe one physical ordering. It records the order the user chose, then allows semantic groups to be layered on top.


## Example 1 — Constant-acceleration state ordered by axis


In [20]:
x_axis_first = StateVector(
    x=(0.0, "m"),
    vx=(10.0, "m/s"),
    ax=(0.0, "m/s^2"),
    y=(0.0, "m"),
    vy=(5.0, "m/s"),
    ay=(0.0, "m/s^2"),
)

x_axis_first.add_group("position", ("x", "y"))
x_axis_first.add_group("velocity", ("vx", "vy"))
x_axis_first.add_group("acceleration", ("ax", "ay"))

print("names:", x_axis_first.names)
print("position indices:", x_axis_first.indices("position"))
print("velocity indices:", x_axis_first.indices("velocity"))
print("acceleration indices:", x_axis_first.indices("acceleration"))


names: ('x', 'vx', 'ax', 'y', 'vy', 'ay')
position indices: (0, 3)
velocity indices: (1, 4)
acceleration indices: (2, 5)


## Example 2 — Constant-acceleration state ordered by physical quantity


In [21]:
x_quantity_first = StateVector(
    x=(0.0, "m"),
    y=(0.0, "m"),
    vx=(10.0, "m/s"),
    vy=(5.0, "m/s"),
    ax=(0.0, "m/s^2"),
    ay=(0.0, "m/s^2"),
)

x_quantity_first.add_group("position", ("x", "y"))
x_quantity_first.add_group("velocity", ("vx", "vy"))
x_quantity_first.add_group("acceleration", ("ax", "ay"))

print("names:", x_quantity_first.names)
print("position indices:", x_quantity_first.indices("position"))
print("velocity indices:", x_quantity_first.indices("velocity"))
print("acceleration indices:", x_quantity_first.indices("acceleration"))


names: ('x', 'y', 'vx', 'vy', 'ax', 'ay')
position indices: (0, 1)
velocity indices: (2, 3)
acceleration indices: (4, 5)


Both constant-acceleration orderings are valid. The scalar storage order differs, so the group indices differ. The user remains responsible for arranging later `F`, `Q`, `H`, and `R` matrices consistently with the chosen order.


## Example 3 — Quaternion attitude and body rates


In [22]:
attitude_q = StateVector(
    q1=(0.0, "1"),
    q2=(0.0, "1"),
    q3=(0.0, "1"),
    q4=(1.0, "1"),
    wx=(0.0, "rad/s"),
    wy=(0.0, "rad/s"),
    wz=(0.0, "rad/s"),
)

attitude_q.add_group(
    "attitude",
    ("q1", "q2", "q3", "q4"),
    representation="quaternion_jpl_scalar_last",
)

attitude_q.add_group(
    "body_rate",
    ("wx", "wy", "wz"),
)

print("attitude representation:", attitude_q.group("attitude").representation)
print("attitude indices:", attitude_q.indices("attitude"))
print("body_rate indices:", attitude_q.indices("body_rate"))


attitude representation: quaternion_jpl_scalar_last
attitude indices: (0, 1, 2, 3)
body_rate indices: (4, 5, 6)


The quaternion representation is declared at the group level, not the scalar level. Scalars `q1`, `q2`, `q3`, and `q4` are just storage components until the `attitude` group gives them geometric meaning.


## Example 4 — MRP attitude and body rates


In [23]:
attitude_mrp = StateVector(
    s1=(0.0, "1"),
    s2=(0.0, "1"),
    s3=(0.0, "1"),
    wx=(0.0, "rad/s"),
    wy=(0.0, "rad/s"),
    wz=(0.0, "rad/s"),
)

attitude_mrp.add_group(
    "attitude",
    ("s1", "s2", "s3"),
    representation="mrp",
)

attitude_mrp.add_group(
    "body_rate",
    ("wx", "wy", "wz"),
)

print("attitude representation:", attitude_mrp.group("attitude").representation)
print("attitude indices:", attitude_mrp.indices("attitude"))
print("body_rate indices:", attitude_mrp.indices("body_rate"))


attitude representation: mrp
attitude indices: (0, 1, 2)
body_rate indices: (3, 4, 5)


## Example 5 — Euler 321 attitude and body rates


In [24]:
attitude_euler = StateVector(
    roll=(0.0, "rad"),
    pitch=(0.0, "rad"),
    yaw=(0.0, "rad"),
    wx=(0.0, "rad/s"),
    wy=(0.0, "rad/s"),
    wz=(0.0, "rad/s"),
)

attitude_euler.add_group(
    "attitude",
    ("roll", "pitch", "yaw"),
    representation="euler_321",
)

attitude_euler.add_group(
    "body_rate",
    ("wx", "wy", "wz"),
)

print("attitude representation:", attitude_euler.group("attitude").representation)
print("attitude indices:", attitude_euler.indices("attitude"))
print("body_rate indices:", attitude_euler.indices("body_rate"))


attitude representation: euler_321
attitude indices: (0, 1, 2)
body_rate indices: (3, 4, 5)


# Section 8 — Unit Testing

This section uses one notebook cell per test case. Each cell can be run independently while developing the architecture. Successful behavior prints a short confirmation; expected failures print the exception type and message.


In [25]:
# Test 1: valid scalar construction.
valid_field = StateField("x", 1, "m")
assert valid_field == StateField("x", 1.0, "m")
print("Test 1 passed: valid scalar construction works.")


Test 1 passed: valid scalar construction works.


In [26]:
# Test 2: unitless construction uses "unspecified".
unitless = StateVector(x=1.0, y=2.0)
assert unitless.units == ("unspecified", "unspecified")
assert unitless.unit("x") == "unspecified"
print("Test 2 passed: omitted units become 'unspecified'.")


Test 2 passed: omitted units become 'unspecified'.


In [27]:
# Test 3: invalid component names fail.
try:
    StateVector(**{"x-position": 1.0})
except Exception as exc:
    print(f"Invalid component name raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Invalid component name should fail.")

try:
    StateVector(**{"class": 1.0})
except Exception as exc:
    print(f"Keyword component name raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Keyword component name should fail.")

print("Test 3 passed: invalid component names are rejected.")


Invalid component name raises ValueError: StateField name must be a valid Python identifier; received 'x-position'.
Keyword component name raises ValueError: StateField name must not be a Python keyword; received 'class'.
Test 3 passed: invalid component names are rejected.


In [28]:
# Test 4: invalid units fail.
try:
    StateVector(x=(1.0, ""))
except Exception as exc:
    print(f"Empty unit raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Empty unit should fail.")

try:
    StateVector(x=(1.0, 1))
except Exception as exc:
    print(f"Non-string unit raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Non-string unit should fail.")

print("Test 4 passed: invalid units are rejected.")


Empty unit raises ValueError: Unit for StateField 'x' must be a non-empty string.
Non-string unit raises TypeError: Unit for StateField 'x' must be a string; received int.
Test 4 passed: invalid units are rejected.


In [29]:
# Test 5: invalid values fail.
try:
    StateVector(x="1.0")
except Exception as exc:
    print(f"String value raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("String value should fail.")

try:
    StateVector(x=[1.0])
except Exception as exc:
    print(f"List value raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("List value should fail.")

try:
    StateVector(x=1.0 + 2.0j)
except Exception as exc:
    print(f"Complex value raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Complex value should fail.")

print("Test 5 passed: invalid values are rejected.")


String value raises TypeError: StateField value for 'x' must be a real numeric scalar; received str.
List value raises TypeError: StateField value for 'x' must be a real numeric scalar; received list.
Complex value raises TypeError: StateField value for 'x' must be a real numeric scalar; received complex.
Test 5 passed: invalid values are rejected.


In [30]:
# Test 6: bool values fail.
try:
    StateVector(x=True)
except Exception as exc:
    print(f"Bool component value raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Bool component value should fail.")

try:
    StateField("flag", False, "1")
except Exception as exc:
    print(f"Bool StateField value raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Bool StateField value should fail.")

print("Test 6 passed: bool values are rejected.")


Bool component value raises TypeError: StateField value for 'x' must not be bool.
Bool StateField value raises TypeError: StateField value for 'flag' must not be bool.
Test 6 passed: bool values are rejected.


In [31]:
# Test 7: deterministic ordering.
state = StateVector(
    x=(1.0, "m"),
    vx=(0.0, "m/s"),
    y=(2.0, "m"),
    vy=(0.0, "m/s"),
)
assert state.names == ("x", "vx", "y", "vy")
np.testing.assert_allclose(state.values, np.array([1.0, 0.0, 2.0, 0.0]))
print("Test 7 passed: declaration order defines numerical order.")


Test 7 passed: declaration order defines numerical order.


In [32]:
# Test 8: scalar access.
assert state["vx"] == 0.0
assert state.index("y") == 2
assert state.indices("y") == (2,)
assert state.unit("vy") == "m/s"
print("Test 8 passed: scalar accessors work.")


Test 8 passed: scalar accessors work.


In [33]:
# Test 9: integer and slice access.
assert state[0] == 1.0
np.testing.assert_allclose(state[0:2], np.array([1.0, 0.0]))

try:
    state[-1]
except Exception as exc:
    print(f"Negative integer index raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Negative integer index should fail.")

try:
    state[True]
except Exception as exc:
    print(f"Bool index raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Bool index should fail.")

print("Test 9 passed: integer and slice access behave correctly.")


Negative integer index raises IndexError: StateVector index -1 is out of range for dimension 4.
Bool index raises TypeError: Boolean indexing is not supported by StateVector.
Test 9 passed: integer and slice access behave correctly.


In [34]:
# Test 10: group creation.
state.add_group("velocity", ("vx", "vy"))
assert state.group("velocity") == StateGroup("velocity", ("vx", "vy"), (1, 3))
assert isinstance(state.group("velocity"), StateGroup)
print("Test 10 passed: group creation stores a StateGroup.")


Test 10 passed: group creation stores a StateGroup.


In [35]:
# Test 11: non-contiguous group access.
state.add_group("position", ("x", "y"))
assert state.indices("position") == (0, 2)
np.testing.assert_allclose(state["position"], np.array([1.0, 2.0]))
print("Test 11 passed: non-contiguous groups work.")


Test 11 passed: non-contiguous groups work.


In [36]:
# Test 12: unknown group/component errors.
try:
    state["missing"]
except Exception as exc:
    print(f"Unknown item access raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Unknown item access should fail.")

try:
    state.indices("missing")
except Exception as exc:
    print(f"Unknown indices lookup raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Unknown indices lookup should fail.")

try:
    state.group("missing")
except Exception as exc:
    print(f"Unknown group lookup raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Unknown group lookup should fail.")

print("Test 12 passed: unknown names fail clearly.")


Unknown item access raises KeyError: "Unknown component or group 'missing'. Available components: ('x', 'vx', 'y', 'vy'). Available groups: ('velocity', 'position')."
Unknown indices lookup raises KeyError: "Unknown component or group 'missing'. Available components: ('x', 'vx', 'y', 'vy'). Available groups: ('velocity', 'position')."
Unknown group lookup raises KeyError: "Unknown component or group 'missing'. Available components: ('x', 'vx', 'y', 'vy'). Available groups: ('velocity', 'position')."
Test 12 passed: unknown names fail clearly.


In [37]:
# Test 13: group name conflict with component name.
try:
    state.add_group("x", ("x", "y"))
except Exception as exc:
    print(f"Group/component name conflict raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Group/component name conflict should fail.")

print("Test 13 passed: group names cannot conflict with component names.")


Group/component name conflict raises ValueError: Group name 'x' conflicts with an existing scalar component.
Test 13 passed: group names cannot conflict with component names.


In [38]:
# Test 14: quaternion representation dimension validation.
StateGroup("attitude", ("q1", "q2", "q3", "q4"), (0, 1, 2, 3), "quaternion_jpl_scalar_last")

try:
    StateGroup("bad_q", ("q1", "q2", "q3"), (0, 1, 2), "quaternion_hamilton_scalar_first")
except Exception as exc:
    print(f"Invalid quaternion dimension raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Invalid quaternion dimension should fail.")

print("Test 14 passed: quaternion dimensions are checked.")


Invalid quaternion dimension raises ValueError: Quaternion StateGroup 'bad_q' must have dimension 4; received dimension 3.
Test 14 passed: quaternion dimensions are checked.


In [39]:
# Test 15: MRP representation dimension validation.
StateGroup("attitude_mrp", ("s1", "s2", "s3"), (0, 1, 2), "mrp")

try:
    StateGroup("bad_mrp", ("s1", "s2", "s3", "s4"), (0, 1, 2, 3), "mrp")
except Exception as exc:
    print(f"Invalid MRP dimension raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Invalid MRP dimension should fail.")

print("Test 15 passed: MRP dimensions are checked.")


Invalid MRP dimension raises ValueError: MRP StateGroup 'bad_mrp' must have dimension 3; received dimension 4.
Test 15 passed: MRP dimensions are checked.


In [40]:
# Test 16: Euler representation dimension validation.
StateGroup("attitude_euler", ("roll", "pitch", "yaw"), (0, 1, 2), "euler_321")

try:
    StateGroup("bad_euler", ("roll", "pitch"), (0, 1), "euler_321")
except Exception as exc:
    print(f"Invalid Euler dimension raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Invalid Euler dimension should fail.")

print("Test 16 passed: Euler dimensions are checked.")


Invalid Euler dimension raises ValueError: Euler StateGroup 'bad_euler' must have dimension 3; received dimension 2.
Test 16 passed: Euler dimensions are checked.


In [41]:
# Test 17: with_values shape and value validation.
updated = state.with_values(np.array([10.0, 1.0, 20.0, 2.0]))
np.testing.assert_allclose(updated.values, np.array([10.0, 1.0, 20.0, 2.0]))

try:
    state.with_values(np.array([1.0, 2.0]))
except Exception as exc:
    print(f"Bad with_values shape raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Bad with_values shape should fail.")

try:
    state.with_values(np.array([1.0, True, 2.0, 0.0], dtype=object))
except Exception as exc:
    print(f"Bad with_values entry raises {type(exc).__name__}: {exc}")
else:
    raise AssertionError("Bad with_values entry should fail.")

print("Test 17 passed: with_values validates replacement values.")


Bad with_values shape raises ValueError: values must have shape (4,); received shape (2,).
Bad with_values entry raises TypeError: values entry 1 must not be bool.
Test 17 passed: with_values validates replacement values.


In [42]:
# Test 18: with_values preserves names, units, groups, and ordering.
assert updated.names == state.names
assert updated.units == state.units
assert tuple(updated.groups) == tuple(state.groups)
assert updated.group("position") == state.group("position")
assert updated.group("velocity") == state.group("velocity")
print("Test 18 passed: with_values preserves state metadata.")


Test 18 passed: with_values preserves state metadata.


In [43]:
# Test 19: values returns a copy.
values_copy = updated.values
values_copy[0] = -999.0
assert updated["x"] == 10.0
print("Test 19 passed: values returns copy-safe storage.")


Test 19 passed: values returns copy-safe storage.


In [44]:
# Test 20: groups returns a shallow copy of the group mapping.
groups_copy = updated.groups
groups_copy["fake"] = StateGroup("fake", ("x",), (0,))
assert "fake" not in updated.groups

print("Test 20 passed: groups returns a shallow copy.")
print("All StateVector architecture tests passed.")


Test 20 passed: groups returns a shallow copy.
All StateVector architecture tests passed.


# Section 9 — Design Review and Next Steps

This notebook established a deterministic state-vector contract. The user-declared component order is explicit, scalar storage is separated from semantic grouping, non-contiguous groups are allowed, and representation metadata is attached at the group level.

The resulting API is ergonomic in Python while still compatible with later fixed-index C/C++ thinking. A Python user can declare a state in natural keyword order, and a lower-level implementation can still see the exact scalar layout and indices.

Main design decisions:

1. `StateField` is internal and canonical. Users normally create fields indirectly through `StateVector(...)`.
2. `StateVector(...)` keyword order defines numerical order.
3. Units are metadata and are not automatically converted.
4. Group representations are strings for now.
5. Non-Euclidean operations are deferred.
6. `StateVector` does not infer physics.
7. `StateGroup` is composed into `StateVector`; `add_group` registers a group, while `group` retrieves the stored group metadata.
8. Public vector and matrix validators are deferred because those contracts belong closer to process and measurement model design.
9. The user remains responsible for constructing `F`, `Q`, `H`, and `R` consistently with the chosen state order once those matrices exist in later notebooks.

Known deferred decisions to resolve later:

- Unit conversion policy: this notebook records units but does not decide whether future code should enforce or convert compatible units.
- Representation typing: strings are simple and notebook-friendly, but future versions may want representation classes or enums.
- Overlapping groups: this notebook allows overlap because groups are semantic views; later model APIs may need rules for when overlap is safe.
- Non-finite values: this notebook rejects `NaN` and infinity for state values and replacement values. If future workflows need placeholders for missing data, that policy should be revisited explicitly.
- Matrix validation location: shape checks for `F`, `Q`, `H`, and `R` should probably live beside model or filter contracts, not in this first state-vector notebook.
- Validation organization: this notebook keeps validation inside the classes for readability. If the same validation rules are reused across future modules, they can be extracted into shared utilities later.

Next notebooks may define simple linear process models or matrix construction helpers. Later notebooks may introduce covariance containers, a linear Kalman filter, EKF machinery, measurement models, representation classes, quaternion/MRP correction operations, and eventually Python-to-C++ layout export ideas.

```text
The StateVector architecture does not solve the estimation problem by itself.
It defines the state contract that every estimator, model, and simulation component must respect.
```
